# WSF Tracker — Year-Code Sensitivity Analysis

**Purpose:** Three analyses of how WSF Tracker temporal choices affect building-dataset validation accuracy:

1. **Section 1 — Sensitivity sweep:** re-run WSF raster validation across a range of `as_of_code` cutoffs (2019 → mid-2025) to quantify how sensitive accuracy metrics are to the chosen year.
2. **Section 2 — Urban growth rate vs. F1:** test whether cities with faster WSF urban growth between the reference image year and mid-2025 show systematically lower vector F1 scores.
3. **Section 3 — Temporally-aligned validation:** re-run WSF raster validation for each city using the cutoff closest to that city's reference imagery date, giving a fairer per-city accuracy estimate free of temporal mismatch.

**WSF year-code encoding (bi-annual):**
`code = round((year − 2016) × 2)`, range [1, 20].
Code 1 = mid-2016, code 2 ≈ end-2016, …, code 19 ≈ mid-2025, code 20 ≈ end-2025.

**Outputs:**
- `outputs/wsf_year_sensitivity.csv` — per-city metrics for each year cutoff (Section 1)
- `outputs/figures/growth_rate_vs_f1.png` — scatter plot (Section 2)
- `outputs/wsf_growth_vs_f1_correlations.csv` — correlation table (Section 2)
- `outputs/wsf_aligned_validation.csv` — per-city temporally-aligned results (Section 3)

Created by: Caroline Gevaert — The World Bank
Financed by: The Gates Foundation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.windows import from_bounds, Window
from rasterio.transform import Affine
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling
import yaml
import warnings

# Suppress Colab/Jupyter's own utcnow deprecation warning — it comes from
# jupyter_client internals and fires on every kernel message; not actionable.
warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow",
    category=DeprecationWarning,
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation"
)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# WSF Tracker bi-annual encoding: code = round((year - 2016) * 2), range [1, 20].
# Odd codes = mid-year; even codes = year-end / next-year-start.
YEAR_CODE_MAP = {
     1: 2016.5,  2: 2017.0,  3: 2017.5,  4: 2018.0,  5: 2018.5,
     6: 2019.0,  7: 2019.5,  8: 2020.0,  9: 2020.5, 10: 2021.0,
    11: 2021.5, 12: 2022.0, 13: 2022.5, 14: 2023.0, 15: 2023.5,
    16: 2024.0, 17: 2024.5, 18: 2025.0, 19: 2025.5, 20: 2026.0,
}

code_to_year = YEAR_CODE_MAP
year_to_code = {v: k for k, v in code_to_year.items()}

BASELINE_CODE = 19  # current config value in validation_configs.yaml (≈ mid-2025)

# Sweep: even codes (integer years 2019–2026) plus the config baseline (code 19)
# so the baseline always appears in the sensitivity summary table.
START_YEAR = 2019
END_YEAR   = max(code_to_year.values())   # 2026.0

year_sweep_base = [y for y in range(int(START_YEAR), int(END_YEAR) + 1) if y in year_to_code]
SWEEP_CODES = sorted(set([year_to_code[y] for y in year_sweep_base] + [BASELINE_CODE]))
SWEEP_YEARS = [code_to_year[c] for c in SWEEP_CODES]

print(f"Encoding: codes 1–{max(code_to_year.keys())}, "
      f"years {min(code_to_year.values())}–{max(code_to_year.values())}")
print(f"Baseline: as_of_code={BASELINE_CODE} → {code_to_year[BASELINE_CODE]} (mid-2025)")
print(f"\nSweep ({len(SWEEP_CODES)} cutoffs):")
for code, year in zip(SWEEP_CODES, SWEEP_YEARS):
    marker = "  ← config baseline" if code == BASELINE_CODE else ""
    print(f"  as_of_code={code:3d}  →  {year}{marker}")

## Section 1 — Year-code sensitivity sweep

For each `as_of_code` in the sweep, re-runs the WSF raster validation across all cities in the AOI tracker and records per-city accuracy metrics. The summary table flags any cutoff that differs from the current pipeline baseline (`as_of_code: 19`) by more than 0.05 F1.

In [ ]:
# ---- Shared helpers ----

def _pixel_area_from_transform(transform) -> float:
    return float(abs(transform.a * transform.e))

def open_in_target_crs(src, target_crs: str):
    if src.crs is None:
        raise ValueError("Raster has no CRS.")
    if str(src.crs) == str(target_crs):
        return src
    return WarpedVRT(src, crs=target_crs, resampling=Resampling.nearest)

def _load_multi_file(base_dir: Path, file_spec: str, crs: str):
    """
    Load one or more pipe-separated vector files from base_dir.
    Returns (GeoDataFrame, list_of_missing_filenames).
    GeoDataFrame is None when no file could be found.
    """
    names = [n.strip() for n in str(file_spec).split("|") if n.strip()]
    parts, missing = [], []
    for name in names:
        p = base_dir / name
        if p.exists():
            parts.append(gpd.read_file(p).to_crs(crs))
        else:
            missing.append(name)
    if not parts:
        return None, missing
    gdf = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
    return gdf, missing

def _read_tile(ds, win, fill):
    """Read a window from ds, working around WarpedVRT's boundless=True restriction."""
    if not isinstance(ds, WarpedVRT):
        return ds.read(1, window=win, boundless=True, fill_value=fill)
    # WarpedVRT forbids boundless reads — clamp to valid extent and pad the rest.
    out_h = max(1, int(round(win.height)))
    out_w = max(1, int(round(win.width)))
    arr   = np.full((out_h, out_w), fill, dtype=ds.dtypes[0])
    r0 = max(0, int(win.row_off))
    c0 = max(0, int(win.col_off))
    r1 = min(ds.height, int(round(win.row_off + win.height)))
    c1 = min(ds.width,  int(round(win.col_off + win.width)))
    if r1 > r0 and c1 > c0:
        dr = max(0, r0 - int(win.row_off))
        dc = max(0, c0 - int(win.col_off))
        patch_h = min(r1 - r0, out_h - dr)
        patch_w = min(c1 - c0, out_w - dc)
        patch = ds.read(
            1,
            window=Window(c0, r0, c1 - c0, r1 - r0),
            out_shape=(patch_h, patch_w),
        )
        arr[dr:dr + patch_h, dc:dc + patch_w] = patch
    return arr

def rasterize_ref_fraction(ref_geoms, out_shape, transform, oversample=4) -> np.ndarray:
    if len(ref_geoms) == 0:
        return np.zeros(out_shape, dtype="float32")
    if oversample <= 1:
        mask = features.rasterize(
            [(g, 1) for g in ref_geoms], out_shape=out_shape,
            transform=transform, fill=0, dtype="uint8"
        )
        return mask.astype("float32")
    h, w = out_shape
    oh, ow = h * oversample, w * oversample
    hi_transform = transform * Affine.scale(1.0 / oversample, 1.0 / oversample)
    hi = features.rasterize(
        [(g, 1) for g in ref_geoms], out_shape=(oh, ow),
        transform=hi_transform, fill=0, dtype="uint8"
    ).astype("float32")
    return hi.reshape(h, oversample, w, oversample).mean(axis=(1, 3)).astype("float32")

def aoi_mask_for_window(aoi_geom, out_shape, transform) -> np.ndarray:
    mask = features.rasterize(
        [(aoi_geom, 1)], out_shape=out_shape,
        transform=transform, fill=0, dtype="uint8"
    )
    return mask.astype(bool)

def wsf_built_pixels(arr: np.ndarray, as_of_code: int,
                     built_value_min: int = 1, nonbuilt_value: int = 0) -> np.ndarray:
    """Binary mask: pixels first detected at or before as_of_code."""
    return (arr != nonbuilt_value) & (arr >= built_value_min) & (arr <= as_of_code)

In [ ]:
# ---- Load AOI tracker to enumerate cities ----
# TODO: update paths if your layout differs from the standard project structure

CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"
TRACKER_PATH = PROJECT_ROOT / "data/02_interim/aoi_tracker.csv"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Use PROJECT_ROOT (hardcoded above) as the base — not cfg["root_dir"], which
# may point to a stale drive path that no longer matches the actual mount location.
DATA_DIR = PROJECT_ROOT / cfg.get("data_dir", "data/01_raw")

# validation_configs.yaml has no top-level 'crs' key.
# Use Web Mercator (EPSG:3857): projected in metres, works globally.
# The main pipeline uses a per-city UTM zone; EPSG:3857 is an acceptable
# approximation for this sensitivity-analysis notebook.
CRS = "EPSG:3857"

TAU_FRAC   = 0.2  # ~20 m² / 100 m² for 10 m pixels; matches main pipeline default
OVERSAMPLE = int(cfg.get("raster", {}).get("preprocessing", {}).get("oversample_factor", 4))

# WSF-specific config — dataset name changed from 'wsf-tracker' (hyphen) to
# 'wsf_tracker' (underscore) in the upstream restructure.
wsf_cfg = next(
    (d for d in cfg["raster"]["datasets"] if d["name"] == "wsf_tracker"),
    None,
)
if wsf_cfg is None:
    avail = [d["name"] for d in cfg["raster"]["datasets"]]
    raise ValueError(
        f"No 'wsf_tracker' entry found under raster.datasets in validation_configs.yaml.\n"
        f"Available dataset names: {avail}"
    )

WSF_BUILT_MIN = int(wsf_cfg["binarize"].get("built_value_min", 1))
WSF_NONBUILT  = int(wsf_cfg["binarize"].get("nonbuilt_value", 0))
# Raster subdirectory name on disk; later cells fall back to the hyphen form
# if the directory was created before the upstream rename.
WSF_DIR_NAME  = wsf_cfg["name"]   # "wsf_tracker"

tracker = pd.read_csv(TRACKER_PATH, dtype=str)
tracker.columns = tracker.columns.str.strip()
tracker = tracker.apply(lambda c: c.str.strip() if c.dtype == object else c)

# Suitable-column name varies across tracker versions; match flexibly.
suitable_col = next((c for c in tracker.columns if "suitable" in c.lower()), None)
if suitable_col:
    tracker = tracker[tracker[suitable_col].str.lower() == "yes"]
else:
    print("⚠  No 'Suitable' column found in tracker — keeping all rows.")

print(f"Cities in tracker (suitable): {tracker['Dataset code'].nunique()}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"DATA_DIR exists? {DATA_DIR.exists()}")
print(f"CRS: {CRS}  |  TAU_FRAC: {TAU_FRAC}  |  OVERSAMPLE: {OVERSAMPLE}")
print(f"WSF built_value_min={WSF_BUILT_MIN}  nonbuilt_value={WSF_NONBUILT}  dir_name='{WSF_DIR_NAME}'")

In [ ]:
# ---- Per-city evaluation ----

def _find_wsf_raster(data_dir, folder: str, wsf_dir_name: str):
    """
    Locate the WSF raster for one city.

    Tries in order:
      1. Flat slug-prefixed file: raster/{slug}_{wsf_dir_name}.tif   (standard layout)
      2. Glob in raster/ for any *{wsf_dir_name}*.tif
      3. Legacy subdirectory: raster/{wsf_dir_name}/*.tif  (or hyphen form)
    Returns sorted list of candidate paths (empty if none found).
    """
    slug = folder.replace("-", "_").replace(" ", "_")
    raster_dir = data_dir / folder / "raster"

    direct = raster_dir / f"{slug}_{wsf_dir_name}.tif"
    if direct.exists():
        return [direct]

    flat = sorted(raster_dir.glob(f"*{wsf_dir_name}*.tif"))
    if flat:
        return flat

    for sub in [wsf_dir_name, wsf_dir_name.replace("_", "-")]:
        sub_dir = raster_dir / sub
        if sub_dir.exists():
            found = sorted(sub_dir.glob("*.tif"))
            if found:
                return found

    return []


def eval_wsf_for_city_all_codes(city: str, row: pd.Series, codes: list) -> list:
    """
    Run WSF validation for one city across all given as_of_code cutoffs in a
    single raster pass.

    Each city's AOI, reference, and WSF raster are opened once. Inside the tile
    loop, reference rasterization is computed once per tile; the per-cutoff
    threshold (arr <= code) is a cheap numpy comparison on the already-loaded
    array. This is ~n_cutoffs× faster than re-opening files per cutoff.

    Returns a list of metric dicts (one per code with valid coverage), or an
    empty list if any required data file is missing.
    aoi_file_name / reference_file_name may be pipe-separated lists of files.
    """
    folder   = str(row["dataset_folder_name"]).strip()
    aoi_spec = str(row.get("aoi_file_name", "")).strip()
    ref_spec = str(row.get("reference_file_name", "")).strip()

    wsf_candidates = _find_wsf_raster(DATA_DIR, folder, WSF_DIR_NAME)

    if not aoi_spec:
        warnings.warn(f"{city}: no AOI filename in tracker")
        return []
    if not ref_spec:
        warnings.warn(f"{city}: no reference filename in tracker")
        return []
    if not wsf_candidates:
        warnings.warn(f"{city}: no WSF raster found under {DATA_DIR / folder / 'raster'}")
        return []

    aoi_gdf, missing_aoi = _load_multi_file(DATA_DIR / folder / "aoi", aoi_spec, CRS)
    if aoi_gdf is None:
        warnings.warn(f"{city}: AOI file(s) not found — {', '.join(missing_aoi)}")
        return []

    ref_gdf, missing_ref = _load_multi_file(DATA_DIR / folder / "vector", ref_spec, CRS)
    if ref_gdf is None:
        warnings.warn(f"{city}: reference file(s) not found — {', '.join(missing_ref)}")
        return []

    wsf_path   = wsf_candidates[0]
    aoi_union  = aoi_gdf.geometry.union_all()
    ref_sindex = ref_gdf.sindex

    from shapely.geometry import box
    minx, miny, maxx, maxy = aoi_union.bounds
    tile_size = float(cfg.get("vector", {}).get("preprocessing", {}).get("tile_size_m", 1000))
    xs = np.arange(minx, maxx, tile_size)
    ys = np.arange(miny, maxy, tile_size)
    tiles = [
        box(x, y, x + tile_size, y + tile_size)
        for x in xs for y in ys
        if box(x, y, x + tile_size, y + tile_size).intersects(aoi_union)
    ]

    # Per-code accumulators — keyed by code
    accum = {c: {"tp": 0.0, "fp": 0.0, "fn": 0.0, "valid": 0.0} for c in codes}

    with rasterio.open(wsf_path) as src:
        ds = open_in_target_crs(src, CRS)
        nodata = ds.nodata

        for tile_geom in tiles:
            win = from_bounds(*tile_geom.bounds, transform=ds.transform)
            if win.width <= 0 or win.height <= 0:
                continue

            # Read tile once — shared across all cutoffs
            arr = _read_tile(ds, win, fill=nodata if nodata is not None else 0)
            transform = rasterio.windows.transform(win, ds.transform)
            pixel_area = _pixel_area_from_transform(transform)

            aoi_mask = aoi_mask_for_window(aoi_union, arr.shape, transform)
            valid = aoi_mask.copy()
            if nodata is not None:
                valid &= (arr != nodata)

            n_valid = int(valid.sum())
            if n_valid == 0:
                continue

            # Rasterize reference once per tile — same for all cutoffs
            possible  = list(ref_sindex.intersection(tile_geom.bounds))
            ref_tile  = ref_gdf.iloc[possible]
            ref_tile  = ref_tile[ref_tile.intersects(tile_geom)]
            f_ref     = rasterize_ref_fraction(
                list(ref_tile.geometry), arr.shape, transform, oversample=OVERSAMPLE
            )
            ref_bin = (f_ref >= TAU_FRAC)[valid]

            # Apply each cutoff threshold — cheap numpy comparison on loaded arr
            for code in codes:
                pred_bin = wsf_built_pixels(arr, code, WSF_BUILT_MIN, WSF_NONBUILT)[valid]
                a = accum[code]
                a["tp"]    += float(np.logical_and( pred_bin,  ref_bin).sum()) * pixel_area
                a["fp"]    += float(np.logical_and( pred_bin, ~ref_bin).sum()) * pixel_area
                a["fn"]    += float(np.logical_and(~pred_bin,  ref_bin).sum()) * pixel_area
                a["valid"] += n_valid * pixel_area

    results = []
    for code in codes:
        a = accum[code]
        if a["valid"] == 0:
            continue
        tp, fp, fn = a["tp"], a["fp"], a["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        results.append({
            "city":                city,
            "as_of_code":          code,
            "as_of_year":          code_to_year.get(code, None),
            "precision_area":      round(precision, 4),
            "recall_area":         round(recall, 4),
            "f1_area":             round(f1, 4),
            "tp_m2":               round(tp, 1),
            "fp_m2":               round(fp, 1),
            "fn_m2":               round(fn, 1),
            "signed_area_bias_m2": round(fp - fn, 1),
            "valid_area_m2":       round(a["valid"], 1),
        })
    return results


def eval_wsf_for_city(city: str, row: pd.Series, as_of_code: int) -> dict | None:
    """Single-cutoff wrapper used by Section 3."""
    res = eval_wsf_for_city_all_codes(city, row, [as_of_code])
    return res[0] if res else None

In [ ]:
# ---- Section 1: main sweep (city-first for efficiency) ----
#
# Iterating cities in the outer loop means each city's AOI, reference, and WSF
# raster are opened exactly once. Inside the tile loop, reference rasterization
# is computed once per tile; all n_cutoffs thresholds are applied as cheap numpy
# comparisons on the already-loaded array. Compared to the cutoff-first approach
# this reduces file I/O and rasterization work by ~n_cutoffs× (~9×).

out_path = PROJECT_ROOT / "outputs/wsf_year_sensitivity.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

cities_grouped = list(tracker.groupby("Dataset code"))
n_cities  = len(cities_grouped)
results   = []

skip_permanently: set  = set()
all_issues: list = []

sweep_start = time.time()

for city_idx, (city, group) in enumerate(cities_grouped, 1):
    row = group.iloc[0]
    t0  = time.time()
    try:
        city_results = eval_wsf_for_city_all_codes(city, row, SWEEP_CODES)
        dt = time.time() - t0
        if city_results:
            results.extend(city_results)
            f1s = [r["f1_area"] for r in city_results]
            print(
                f"  [{city_idx}/{n_cities}] {city}: "
                f"{len(city_results)} cutoffs | "
                f"F1 {min(f1s):.3f}–{max(f1s):.3f}  ({dt:.0f}s)"
            )
        else:
            skip_permanently.add(city)
            all_issues.append({"city": city, "reason": "missing data"})
            print(f"  [{city_idx}/{n_cities}] {city}: SKIPPED (missing data)  ({dt:.0f}s)")
    except Exception as exc:
        dt = time.time() - t0
        skip_permanently.add(city)
        all_issues.append({"city": city, "reason": str(exc)})
        print(f"  [{city_idx}/{n_cities}] {city}: ERROR — {exc}  ({dt:.0f}s)")

    # Save after every city — robust to kernel crashes
    if results:
        pd.DataFrame(results).to_csv(out_path, index=False)

    elapsed   = time.time() - sweep_start
    remaining = n_cities - city_idx
    eta_min   = (elapsed / city_idx) * remaining / 60
    print(
        f"    elapsed {elapsed/60:.1f} min | "
        f"ETA ~{eta_min:.0f} min | "
        f"{remaining} cities left, {len(skip_permanently)} skipped"
    )

sensitivity_df = pd.DataFrame(results)
sensitivity_df.to_csv(out_path, index=False)
total_min = (time.time() - sweep_start) / 60
print(f"\nComplete in {total_min:.1f} min — saved → {out_path}")
print(f"Rows: {len(sensitivity_df)}  |  Cities: {sensitivity_df['city'].nunique()}  |  Cutoffs: {sensitivity_df['as_of_year'].nunique()}")

# ---- Issue summary ----
if all_issues:
    print(f"\n{'='*60}")
    print(f"SKIPPED / ERRORED — {len(all_issues)} cities:")
    print(f"{'='*60}")
    for issue in sorted(all_issues, key=lambda x: x["city"]):
        print(f"  {issue['city']:<30}  {issue['reason']}")
else:
    print("\n✓ All cities processed successfully.")

## Section 1 summary — Sensitivity table and flagging

In [ ]:
# ---- Summary: mean F1 and mean bias per cutoff year ----

summary = (
    sensitivity_df
    .groupby(["as_of_year", "as_of_code"])
    .agg(
        n_cities             =("city",                "nunique"),
        mean_f1              =("f1_area",             "mean"),
        median_f1            =("f1_area",             "median"),
        mean_precision       =("precision_area",      "mean"),
        mean_recall          =("recall_area",         "mean"),
        mean_bias_km2        =("signed_area_bias_m2", lambda x: x.mean() / 1e6),
    )
    .reset_index()
    .sort_values("as_of_year")
)

# Flag relative to baseline (as_of_code: 19)
baseline_row = summary[summary["as_of_code"] == 19]
if len(baseline_row) == 1:
    baseline_f1 = float(baseline_row["mean_f1"].iloc[0])
    summary["f1_delta_vs_baseline"] = (summary["mean_f1"] - baseline_f1).round(4)
    summary["flagged"] = summary["f1_delta_vs_baseline"].abs() > 0.05
    print(f"Baseline (as_of_code=19) mean F1: {baseline_f1:.4f}")
else:
    summary["f1_delta_vs_baseline"] = np.nan
    summary["flagged"] = False
    print("Note: as_of_code=19 not in sweep (year 2019 may not be in data range).")

print("\n=== WSF Year-Code Sensitivity Summary ===")
display(
    summary[[
        "as_of_year", "as_of_code", "n_cities",
        "mean_f1", "median_f1", "mean_precision", "mean_recall",
        "mean_bias_km2", "f1_delta_vs_baseline", "flagged"
    ]].round(4)
)

flagged = summary[summary["flagged"]]
if len(flagged):
    print(f"\n⚠  {len(flagged)} cutoff(s) differ from baseline by >0.05 F1:")
    for _, r in flagged.iterrows():
        print(f"   {int(r.as_of_year)} (code={int(r.as_of_code)}): "
              f"mean F1={r.mean_f1:.4f}  delta={r.f1_delta_vs_baseline:+.4f}")
else:
    print("\n✓  No cutoff differs from baseline by more than 0.05 F1.")

In [ ]:
# ---- Interpretation: is as_of_code=19 capturing all years or cutting off early? ----

max_code_in_data = max(code_to_year.keys())
max_year_in_data = code_to_year[max_code_in_data]

print("=" * 55)
print("as_of_code=19 interpretation")
print("=" * 55)

if 19 > max_code_in_data:
    print(f"as_of_code=19 EXCEEDS the maximum code ({max_code_in_data} = {max_year_in_data}).")
    print("→ Current config captures ALL years — no temporal bias introduced.")
elif 19 in code_to_year:
    mapped_year = code_to_year[19]
    # int() handles float years (e.g. 2024.5) when constructing the year range.
    next_full_year = int(mapped_year) + 1
    max_full_year  = int(max_year_in_data)
    years_missing  = list(range(next_full_year, max_full_year + 1))
    print(f"as_of_code=19 → {mapped_year} in the bi-annual encoding.")
    if years_missing:
        print(f"→ Full calendar years excluded from built-up mask: {years_missing}")
        print("→ Buildings added in those years appear as False Negatives.")
        print("→ Consider updating as_of_code in configs/validation_configs.yaml.")
    else:
        print("→ No full calendar years are excluded — gap to max code is < 1 year.")
        print("→ Temporal bias from the current as_of_code is negligible.")
else:
    print(f"as_of_code=19 is not in the confirmed mapping.")
    print(f"Valid codes: {sorted(code_to_year.keys())}")

---

## Section 2: Urban Growth Rate vs. Tile F1 Accuracy

**Purpose:** Test whether cities with faster urban growth (more WSF pixels added between the reference image year and Jan 2025) have systematically lower F1 scores, indicating that temporal mismatch — not just spatial accuracy — is a meaningful driver of validation error.

**Inputs:**
- `Reference dataset overview.xlsx` — maps each city to the year of its reference imagery
- WSF Tracker rasters on disk — used to compute built-up pixel counts at two time points
- `vector_all_cities_merged.xlsx` — per-city F1 scores from the vector validation pipeline

**Method:**
1. For each city, derive `wsf_code_early` from the reference image year using `code = round((ref_year - 2015) × 2)`, clipped to [1, 20].
2. Count built-up WSF pixels at `wsf_code_early` and `wsf_code_late = 19` (Jan 2025).
3. Merge with vector F1. Correlate growth rate with F1 across datasets, split by SpaceNet7 vs. other.

### Cell 1 — Load reference image years

In [ ]:
import warnings

# TODO: update path if the overview file lives elsewhere
REF_OVERVIEW_PATH = PROJECT_ROOT / "data/02_interim/Reference dataset overview.xlsx"

ref_raw = pd.read_excel(
    REF_OVERVIEW_PATH,
    sheet_name="Reference Dataset overview",
    dtype=str,
)
ref_raw.columns = ref_raw.columns.str.strip()

# Identify the two relevant columns (names may have slight whitespace variations)
code_col = next(c for c in ref_raw.columns if "dataset" in c.lower() and "code" in c.lower())
year_col = next(c for c in ref_raw.columns if "year" in c.lower())

ref_year_map: dict[str, int] = {}   # city_code -> int year
skipped_cities: list[str] = []

for _, row in ref_raw.iterrows():
    city = str(row[code_col]).strip()
    raw_year = str(row[year_col]).strip()

    if not city or city.lower() in {"nan", ""}:
        continue

    # Skip rows with "various", missing, or non-numeric year
    if raw_year.lower() in {"various", "nan", "", "n/a", "tbc"}:
        skipped_cities.append(f"{city}  (year='{raw_year}')")
        continue

    try:
        ref_year_map[city] = int(float(raw_year))
    except ValueError:
        skipped_cities.append(f"{city}  (year='{raw_year}' — unparseable)")

print(f"Cities with confirmed reference year: {len(ref_year_map)}")
print(f"Cities skipped:                       {len(skipped_cities)}")

if skipped_cities:
    print("\n⚠ Skipped cities (year missing or 'various'):")
    for s in skipped_cities:
        print(f"   {s}")

print("\nSample ref_year_map (first 10):")
for city, yr in list(ref_year_map.items())[:10]:
    print(f"  {city}: {yr}")

### Cell 2 — WSF growth rate per city

In [ ]:
WSF_CODE_LATE = 19   # mid-2025 (current as_of_code in configs/validation_configs.yaml)
GROWTH_CAP    = 5.0  # 500 % cap to handle near-zero denominators

# bi-annual encoding: code = round((year - 2016) * 2), range [1, 20]
# code 1 = mid-2016, code 2 = end-2016, code 3 = mid-2017, ...
def ref_year_to_wsf_code(year: int) -> int:
    return int(np.clip(round((year - 2016) * 2), 1, 20))

def count_wsf_built_pixels(wsf_path, as_of_code: int,
                            built_value_min: int = 1,
                            nonbuilt_value: int = 0) -> int:
    """Read full raster and count pixels settled at or before as_of_code."""
    with rasterio.open(wsf_path) as src:
        arr = src.read(1)
        nodata = src.nodata
        valid = np.ones(arr.shape, dtype=bool)
        if nodata is not None:
            valid &= (arr != nodata)
        mask = wsf_built_pixels(arr, as_of_code, built_value_min, nonbuilt_value)
        return int((mask & valid).sum())

growth_rows = []

for city, ref_year in ref_year_map.items():
    # Look up dataset folder from tracker
    city_rows = tracker[tracker["Dataset code"] == city]
    if city_rows.empty:
        warnings.warn(f"{city}: not found in AOI tracker — skipping.")
        continue

    folder = str(city_rows.iloc[0]["dataset_folder_name"]).strip()

    # Use the same helper as the sensitivity sweep for consistent path resolution.
    wsf_candidates = _find_wsf_raster(DATA_DIR, folder, WSF_DIR_NAME)
    if not wsf_candidates:
        warnings.warn(f"{city}: no WSF raster found under {DATA_DIR / folder / 'raster'} — skipping.")
        continue

    wsf_path = wsf_candidates[0]
    wsf_code_early = ref_year_to_wsf_code(ref_year)

    try:
        pixels_early = count_wsf_built_pixels(
            wsf_path, wsf_code_early, WSF_BUILT_MIN, WSF_NONBUILT
        )
        pixels_late = count_wsf_built_pixels(
            wsf_path, WSF_CODE_LATE, WSF_BUILT_MIN, WSF_NONBUILT
        )
    except Exception as exc:
        warnings.warn(f"{city}: raster read failed — {exc}")
        continue

    if pixels_early == 0:
        warnings.warn(f"{city}: zero built pixels at code {wsf_code_early} — "
                      f"growth rate undefined; capping at {GROWTH_CAP:.0%}.")
        growth_rate = GROWTH_CAP
    else:
        growth_rate = min((pixels_late - pixels_early) / pixels_early, GROWTH_CAP)

    growth_rows.append({
        "city":               city,
        "ref_image_year":     ref_year,
        "wsf_code_early":     wsf_code_early,
        "wsf_code_late":      WSF_CODE_LATE,
        "temporal_gap_years": (WSF_CODE_LATE - wsf_code_early) / 2.0,
        "pixels_early":       pixels_early,
        "pixels_late":        pixels_late,
        "growth_rate":        round(growth_rate, 4),
    })

growth_df = pd.DataFrame(growth_rows)
print(f"Growth rates computed for {len(growth_df)} cities.")
print(f"Capped at {GROWTH_CAP:.0%}: "
      f"{(growth_df['growth_rate'] == GROWTH_CAP).sum()} city/cities.")

### Cell 3 — Year selection log

> ⚠ **Review this table before interpreting results.**
> Verify that `ref_image_year` matches what you expect from the reference data documentation, and that `wsf_code_early` is the correct bi-annual code for that year. Cities where `temporal_gap_years` is 0 or negative should be investigated — they may have a reference image year that is more recent than the WSF cutoff.

In [ ]:
log_cols = ["city", "ref_image_year", "wsf_code_early", "wsf_code_late", "temporal_gap_years"]
log_df = growth_df[log_cols].sort_values("city").reset_index(drop=True)

print(f"{'city':<25} {'ref_year':>8} {'code_early':>10} {'code_late':>9} {'gap_years':>9}")
print("-" * 65)
for _, r in log_df.iterrows():
    flag = "  ← ⚠" if r.temporal_gap_years <= 0 else ""
    print(
        f"{r.city:<25} {int(r.ref_image_year):>8} {int(r.wsf_code_early):>10} "
        f"{int(r.wsf_code_late):>9} {r.temporal_gap_years:>9.1f}{flag}"
    )

suspicious = log_df[log_df["temporal_gap_years"] <= 0]
if not suspicious.empty:
    print(f"\n⚠ {len(suspicious)} city/cities have zero or negative temporal gap — investigate:")
    print(suspicious.to_string(index=False))

### Cell 4 — OBT temporal encoding

**Finding from pipeline audit (`src/download/raster.py`, `src/validate/raster_runner.py`, `configs/validation_configs.yaml`):**

OBT year selection is **hardcoded to 2023** in the validation config:

```yaml
# configs/validation_configs.yaml
- name: obt
  year: 2023          # ← fixed; raster_runner globs {slug}_obt_2023*.tif
```

**Download side** (`OBTRunner.run()`): loops over `GoogleOBTConfig.years = [2016, …, 2023]` and saves one GeoTIFF per year as `{slug}_obt_{year}.tif`. All annual files are on disk.

**Validation side** (`raster_runner.py` L114–122): uses the `year` field to build the glob pattern `{slug}_obt_2023*` and picks the first match. It is **not** matched dynamically to the reference image year.

**Implication for growth rate analysis:** Computing OBT growth rate requires reading two annual files per city — `{slug}_obt_{ref_year}.tif` (early) and `{slug}_obt_2023.tif` (late). Both files exist on disk for any reference year in [2016, 2023]. This is a straightforward two-file comparison, but it requires explicit year-pair logic not yet in the pipeline.

In [ ]:
# TODO: implement OBT growth rate comparison.
#
# For each city with ref_image_year in [2016, 2023]:
#   early_path = DATA_DIR / folder / "raster" / f"{slug}_obt_{ref_year}.tif"
#   late_path  = DATA_DIR / folder / "raster" / f"{slug}_obt_2023.tif"
#   Compare fractional building coverage at both years (band 1 = building_presence).
#   growth_rate = (mean_presence_late - mean_presence_early) / mean_presence_early
#
# This requires two-file explicit extraction — not yet implemented in the pipeline.
# The annual files already exist on disk (downloaded by OBTRunner).
print("OBT growth rate: TODO — see markdown above for implementation notes.")

### Cell 5 — Merge WSF growth rates with vector F1

In [ ]:
# TODO: update paths if your outputs live elsewhere
VECTOR_MERGED_PATH = PROJECT_ROOT / "outputs/vector_all_cities_merged.xlsx"

# Load vector F1 scores
# Expected columns: city (or Dataset code), dataset (GBA / GlobFP / Overture), f1
# TODO: verify column names match your actual file before running
f1_raw = pd.read_excel(
    VECTOR_MERGED_PATH,
    sheet_name="vector_all_cities_merged",
    dtype=str,
)
f1_raw.columns = f1_raw.columns.str.strip()

# Normalise city column name — the file may use "city", "Dataset code", or similar
city_col_f1 = next(
    (c for c in f1_raw.columns if "city" in c.lower() or "dataset" in c.lower()),
    f1_raw.columns[0],
)
f1_raw = f1_raw.rename(columns={city_col_f1: "city"})

# Identify F1 and dataset columns
dataset_col = next(c for c in f1_raw.columns if "dataset" in c.lower() and c != "city")
f1_col = next(c for c in f1_raw.columns if "f1" in c.lower())

f1_df = f1_raw[["city", dataset_col, f1_col]].copy()
f1_df.columns = ["city", "dataset", "f1"]
f1_df["f1"] = pd.to_numeric(f1_df["f1"], errors="coerce")
f1_df = f1_df.dropna(subset=["f1"])

# Load SpaceNet7 city codes
sn7_raw = pd.read_excel(
    VECTOR_MERGED_PATH,
    sheet_name="new regions",
    dtype=str,
)
sn7_raw.columns = sn7_raw.columns.str.strip()
sn7_col = next(c for c in sn7_raw.columns if "new city" in c.lower() or "city" in c.lower())
sn7_cities = set(sn7_raw[sn7_col].dropna().str.strip().unique())

f1_df["is_spacenet7"] = f1_df["city"].isin(sn7_cities)

# Merge with growth rates
merged_df = f1_df.merge(
    growth_df[["city", "ref_image_year", "wsf_code_early", "temporal_gap_years", "growth_rate"]],
    on="city",
    how="inner",
)

print(f"F1 rows loaded:          {len(f1_df)}")
print(f"After merge with growth: {len(merged_df)}")
print(f"Unique cities in merged: {merged_df['city'].nunique()}")
print(f"SpaceNet7 cities:        {merged_df[merged_df['is_spacenet7']]['city'].nunique()}")
print(f"Datasets present:        {sorted(merged_df['dataset'].unique())}")
display(merged_df.head(10))

### Cell 6 — Scatter plot: growth rate vs. F1, by dataset and group

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

DATASET_COLORS = {
    "GBA":     "#1f77b4",
    "GlobFP":  "#ff7f0e",
    "Overture": "#2ca02c",
}

# Normalise dataset labels (capitalise first letter; adjust if your file uses different casing)
merged_df["dataset_label"] = merged_df["dataset"].str.strip().str.title()

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
group_labels = {True: "SpaceNet7 cities", False: "Non-SpaceNet7 cities"}

for ax, is_sn7 in zip(axes, [True, False]):
    subset = merged_df[merged_df["is_spacenet7"] == is_sn7]
    ax.set_title(group_labels[is_sn7], fontsize=13, fontweight="bold")

    for ds_label, ds_color in DATASET_COLORS.items():
        ds_data = subset[subset["dataset_label"] == ds_label].dropna(
            subset=["growth_rate", "f1"]
        )
        if ds_data.empty:
            continue

        ax.scatter(
            ds_data["growth_rate"],
            ds_data["f1"],
            color=ds_color,
            alpha=0.75,
            s=60,
            zorder=3,
            label=ds_label,
        )

        # Regression line (requires >= 2 points)
        if len(ds_data) >= 2:
            x = ds_data["growth_rate"].values
            y = ds_data["f1"].values
            coeffs = np.polyfit(x, y, 1)
            x_range = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_range, np.polyval(coeffs, x_range),
                    color=ds_color, linewidth=1.5, linestyle="--", alpha=0.8)

    ax.set_xlabel("WSF growth rate (early → Jan 2025)", fontsize=11)
    ax.set_ylabel("Vector F1", fontsize=11)
    ax.set_xlim(left=-0.05)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.axhline(0.5, color="grey", linewidth=0.8, linestyle=":")

    # City labels for outliers (optional — top/bottom 3 per group)
    for _, row in subset.nlargest(3, "growth_rate").iterrows():
        ax.annotate(
            row["city"], (row["growth_rate"], row["f1"]),
            fontsize=7, alpha=0.7,
            xytext=(4, 2), textcoords="offset points",
        )

# Shared legend
handles = [
    mlines.Line2D([], [], color=c, marker="o", linestyle="None", markersize=7, label=d)
    for d, c in DATASET_COLORS.items()
]
fig.legend(handles=handles, title="Dataset", loc="lower center",
           ncol=3, bbox_to_anchor=(0.5, -0.04), frameon=False, fontsize=10)

fig.suptitle(
    "WSF urban growth rate vs. vector F1 accuracy\n"
    "(dashed lines = per-dataset OLS regression)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig(
    PROJECT_ROOT / "outputs/figures/growth_rate_vs_f1.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Figure saved → outputs/figures/growth_rate_vs_f1.png")

### Cell 7 — Correlation table (Pearson & Spearman)

In [ ]:
from scipy import stats

corr_rows = []

for ds_label in merged_df["dataset_label"].unique():
    for is_sn7, group_name in [(True, "SpaceNet7"), (False, "Non-SpaceNet7")]:
        subset = merged_df[
            (merged_df["dataset_label"] == ds_label) &
            (merged_df["is_spacenet7"] == is_sn7)
        ].dropna(subset=["growth_rate", "f1"])

        n = len(subset)
        if n < 4:
            # Too few points for meaningful correlation
            corr_rows.append({
                "dataset": ds_label, "group": group_name, "n": n,
                "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_r": np.nan, "spearman_p": np.nan,
                "note": f"n<4 — skipped",
            })
            continue

        x = subset["growth_rate"].values
        y = subset["f1"].values

        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)

        corr_rows.append({
            "dataset":    ds_label,
            "group":      group_name,
            "n":          n,
            "pearson_r":  round(pr, 3),
            "pearson_p":  round(pp, 4),
            "spearman_r": round(sr, 3),
            "spearman_p": round(sp, 4),
            "note":       "significant (p<0.05)" if min(pp, sp) < 0.05 else "",
        })

corr_df = pd.DataFrame(corr_rows).sort_values(["dataset", "group"]).reset_index(drop=True)

print("=== Pearson & Spearman correlations: growth rate vs. F1 ===\n")
print(corr_df.to_string(index=False))

sig = corr_df[corr_df["note"].str.contains("significant", na=False)]
if not sig.empty:
    print(f"\n✓ Significant correlations (p<0.05 on at least one test):")
    for _, r in sig.iterrows():
        print(f"  {r['dataset']} / {r['group']}: "
              f"Pearson r={r.pearson_r} (p={r.pearson_p}), "
              f"Spearman r={r.spearman_r} (p={r.spearman_p})")
else:
    print("\nNo significant correlations found (p<0.05) — "
          "temporal mismatch alone may not explain F1 variation in this sample.")

# Save table
corr_df.to_csv(PROJECT_ROOT / "outputs/wsf_growth_vs_f1_correlations.csv", index=False)
print("\nSaved → outputs/wsf_growth_vs_f1_correlations.csv")

---

## Section 3: Per-City Temporally-Aligned WSF Validation

**Purpose:** Re-run WSF raster validation for each city using the `as_of_code` that best matches when that city's reference imagery was captured, rather than the global `as_of_code: 19` applied uniformly in the main pipeline.

**Why this matters:** The main pipeline compares every reference dataset against a WSF built-up mask that extends to Jan 2025 regardless of when the imagery was taken. A city with 2020 reference imagery is penalised for urban growth that occurred in 2021–2024 — those new buildings appear as False Positives. Aligning the WSF cutoff to the reference year isolates *spatial* accuracy from *temporal mismatch*.

**Method:** For each city, compute `wsf_code = round((ref_year − 2015) × 2)` and run `eval_wsf_for_city` at that code. The comparison cell then merges with Section 1 baseline results (as_of_code = 19) to show the F1 delta per city.

**Dependencies:** run Section 1 helpers cell (`eval_wsf_for_city`, `tracker`) and Section 2 cells 1–2 (`ref_year_map`, `ref_year_to_wsf_code`) before running this section. Section 1 sweep is optional but needed for the baseline comparison.

In [ ]:
# ---- Section 3: per-city aligned validation loop ----
# Requires: ref_year_map + ref_year_to_wsf_code (Section 2 cells 1-2)
#           eval_wsf_for_city + tracker (Section 1)

import warnings as _w
_w.filterwarnings("always")

aligned_rows    = []
skipped_aligned = []

for city, ref_year in ref_year_map.items():
    city_rows = tracker[tracker["Dataset code"] == city]
    if city_rows.empty:
        skipped_aligned.append(f"{city}: not in tracker")
        continue

    wsf_code = ref_year_to_wsf_code(ref_year)
    row = city_rows.iloc[0]

    try:
        res = eval_wsf_for_city(city, row, as_of_code=wsf_code)
        if res is not None:
            res["ref_image_year"]   = ref_year
            res["wsf_code_aligned"] = wsf_code
            aligned_rows.append(res)
            print(f"  {city} (ref_year={ref_year}, code={wsf_code}): F1={res['f1_area']:.3f}")
        else:
            skipped_aligned.append(f"{city}: missing data")
    except Exception as exc:
        skipped_aligned.append(f"{city}: ERROR — {exc}")
        print(f"  {city}: ERROR — {exc}")

aligned_df = pd.DataFrame(aligned_rows)
print(f"\nAligned validation: {len(aligned_df)} cities  |  {len(skipped_aligned)} skipped")
if skipped_aligned:
    print("Skipped:")
    for s in skipped_aligned:
        print(f"  {s}")

In [ ]:
# ---- Section 3: compare aligned vs. pipeline baseline ----

BASELINE_CODE_S3 = 19

if (
    "sensitivity_df" in dir()
    and not sensitivity_df.empty
    and BASELINE_CODE_S3 in sensitivity_df["as_of_code"].values
):
    baseline_city = (
        sensitivity_df[sensitivity_df["as_of_code"] == BASELINE_CODE_S3]
        [["city", "f1_area", "precision_area", "recall_area"]]
        .rename(columns={
            "f1_area":        "f1_baseline",
            "precision_area": "precision_baseline",
            "recall_area":    "recall_baseline",
        })
    )
    comparison_df = aligned_df.merge(baseline_city, on="city", how="left")
    comparison_df["f1_delta"] = comparison_df["f1_area"] - comparison_df["f1_baseline"]
    has_baseline = True
else:
    comparison_df = aligned_df.copy()
    has_baseline = False
    print("Note: sensitivity_df not available — run Section 1 first to enable baseline comparison.")

print("=== Per-city temporally-aligned WSF validation ===\n")
print(f"Cities validated:  {len(aligned_df)}")
print(f"Mean F1:           {aligned_df['f1_area'].mean():.4f}")
print(f"Mean precision:    {aligned_df['precision_area'].mean():.4f}")
print(f"Mean recall:       {aligned_df['recall_area'].mean():.4f}")

if has_baseline:
    delta = comparison_df["f1_delta"].dropna()
    print(f"\nvs. pipeline baseline (as_of_code={BASELINE_CODE_S3} ≈ {code_to_year[BASELINE_CODE_S3]}):")
    print(f"  Mean F1 delta:          {delta.mean():+.4f}")
    print(f"  Cities improved >+0.01: {(delta >  0.01).sum()}")
    print(f"  Cities degraded >-0.01: {(delta < -0.01).sum()}")

cols_show = ["city", "ref_image_year", "wsf_code_aligned", "f1_area", "precision_area", "recall_area"]
if has_baseline:
    cols_show += ["f1_baseline", "f1_delta"]

display(
    comparison_df[cols_show]
    .sort_values("f1_delta" if has_baseline else "f1_area", ascending=True)
    .reset_index(drop=True)
)

out_path = PROJECT_ROOT / "outputs/wsf_aligned_validation.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")